# 03_02 Logistic regression: a spam filter, measured honestly

The book's notebook trained a logistic regression filter and reported 97 percent accuracy. It also had a
bug that its own test could not see. By the end of this notebook you will have trained the filter, read its
confusion matrix, moved its threshold, drawn its ROC curve, found the bug, and fixed it the way production
systems do.

**How this notebook works.** The same rhythm as every notebook in this course:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-03-teaching-a-machine-what-spam-looks-like", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'sentence_transformers': 'sentence-transformers',
           'nltk': 'nltk',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib',
           'joblib': 'joblib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    import nltk
    for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng', 'maxent_ne_chunker_tab', 'words']:
        nltk.download(pkg, quiet=True)
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn import metrics
from spamtools import load_sms, split, clean
from nlpcheck import ask, guess, reveal, check_03_02

df = load_sms()
X_train, X_test, y_train, y_test = split(df)

## 1. Recall

**r4.** What was the spam recall of the filter that always says ham? (a number)

**r5.** Precision answers which question? (a) of all the spam, how much did I catch?
(b) of the messages I flagged, how many were spam? (c) how many messages did I get right?

In [ ]:
ask("r4", "")
ask("r5", "")

## 2. The worked example: a pipeline

A **pipeline** chains steps so that one object turns raw text into a prediction: here the TF-IDF
vectorizer from Lab 02, then a logistic regression. `fit` learns the vocabulary and the weights from the
training messages only; `predict` applies both to new text.

In [ ]:
lr = make_pipeline(TfidfVectorizer(), LogisticRegression())
lr.fit(X_train, y_train)
pred = lr.predict(X_test)
print(metrics.confusion_matrix(y_test, pred, labels=["ham", "spam"]))
print(metrics.classification_report(y_test, pred, digits=3))

Read the matrix as in the last notebook: rows are the truth, columns the prediction. It flags 182 spam
messages and every flag is right (**precision 1.0**), but it misses 42 (**recall 0.81**). The overall
accuracy is 0.975, which you now know to read as "compared with 0.866 for doing nothing".

## 3. The threshold

Logistic regression does not really answer "spam or ham". It answers "the probability this is spam", and
`predict` flags anything at 0.5 or above. Predict: if you lower the threshold to 0.2, how many of the 42
missed spam messages are still missed?

In [ ]:
spam = list(lr.classes_).index("spam")
p_spam = lr.predict_proba(X_test)[:, spam]
guess("missed_at_0_2", None)   # a number from 0 to 42

In [ ]:
for t in [0.9, 0.5, 0.4, 0.3, 0.2, 0.1]:
    flag = p_spam >= t
    cm = metrics.confusion_matrix(y_test == "spam", flag)
    print(f"threshold {t:.1f}: flagged {flag.sum():4}  missed spam {cm[1, 0]:3}  ham wrongly flagged {cm[0, 1]:4}"
          f"  precision {metrics.precision_score(y_test == 'spam', flag):.3f}  recall {metrics.recall_score(y_test == 'spam', flag):.3f}")
reveal("missed_at_0_2", int(((p_spam < 0.2) & (y_test == "spam")).sum()))

At 0.2 it misses 12 instead of 42, and pays for it with 28 real messages sent to the spam folder. Every
threshold is a different filter built from the same model. The **ROC curve** draws all of them at once: the
true positive rate (recall) against the false positive rate (ham wrongly flagged, as a fraction of all ham),
and the area under it, **AUC**, scores the model without choosing a threshold at all.

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(y_test == "spam", p_spam)
auc = metrics.roc_auc_score(y_test == "spam", p_spam)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"logistic regression, AUC {auc:.3f}")
plt.plot([0, 1], [0, 1], "--", color="grey", label="guessing, AUC 0.5")
plt.xlabel("false positive rate (ham flagged)"); plt.ylabel("true positive rate (spam caught)")
plt.legend(loc="lower right"); plt.title("Every threshold at once");

## 4. Your turn: choose a threshold honestly

Kittiwake's brief for its SMS gateway: **catch at least 90 percent of spam**. You could slide the threshold
along the test set until recall hits 0.90, but then the test set has helped choose the model and is no longer
a fair test. Instead, get probabilities for the **training** messages by cross-validation (each message
scored by a model that did not see it), choose the threshold there, and only then look at the test set.

Fill in the line that picks the threshold: the **highest** threshold whose cross-validated recall is at least
0.90.

In [ ]:
cv_p = cross_val_predict(make_pipeline(TfidfVectorizer(), LogisticRegression()),
                         X_train, y_train, cv=5, method="predict_proba")[:, spam]
candidates = np.round(np.arange(0.99, 0.0, -0.01), 2)
recalls = [metrics.recall_score(y_train == "spam", cv_p >= t) for t in candidates]

threshold = 0.5   # YOUR CODE HERE: the first (highest) candidate whose recall is at least 0.90

flag = p_spam >= threshold
print("threshold", threshold, "| test recall", round(metrics.recall_score(y_test == "spam", flag), 3),
      "| test precision", round(metrics.precision_score(y_test == "spam", flag), 3))

With the right line the threshold is 0.28 and the test recall comes out at 0.911: the promise made on the
training data held on data the choice never saw. That is the whole point of choosing it this way.

## 5. The planted bug: the model that was cleaned one way and served another

This is the book's notebook, faithfully. It cleans the messages with the Chapter 1 steps (`clean`: letters
only, lowercase, stop words out, Porter stems), then trains the pipeline on `X_train`, the **raw** messages.
Later it writes a `serve` function, the thing that would sit behind Kittiwake's SMS gateway, which cleans
each incoming message before asking the model.

In [ ]:
corpus = [clean(m) for m in X_train]          # cleaned ... and then never used
print(X_train.iloc[0], "->", corpus[0])

model = make_pipeline(TfidfVectorizer(), LogisticRegression())
model.fit(X_train, y_train)                   # trained on the raw text

def serve(message):
    return model.predict([clean(message)])[0] # served on cleaned text

The test set says the model catches 81 percent of spam. Predict what fraction of the test spam `serve`
catches, when the same messages arrive at the gateway.

In [ ]:
guess("served_recall", None)   # a fraction

In [ ]:
served = pd.Series([serve(m) for m in X_test], index=X_test.index)
print("model on the test set:", round(metrics.recall_score(y_test, model.predict(X_test), pos_label="spam"), 3))
reveal("served_recall", round(metrics.recall_score(y_test, served, pos_label="spam"), 3))
os.makedirs("out", exist_ok=True)
served.rename("served").rename_axis("message_index").to_csv("out/03_02_served.csv")

0.58. A quarter of the spam the evaluation promised to catch gets through, and no metric in the notebook
shows it, because the evaluation called the model one way and the gateway calls it another. Stemmed text is
full of words the model never saw in training (`receiv`, `entri`), which TF-IDF quietly ignores. This is
called **training/serving skew**, and it is one of the most common ways a model that tested well fails in
production.

**Fix it** so that it cannot happen: put the cleaning *inside* the pipeline, where `fit` and `predict` both
run it. Two edits in the cell above:

- build the vectorizer as `TfidfVectorizer(preprocessor=clean)`,
- and make `serve` pass the raw message: `model.predict([message])[0]`.

Run that cell, then the one after it. `served` should now match the model's own test predictions.

## 6. Save the filter and check

In [ ]:
joblib.dump(model, "out/spam_lr.joblib")
json.dump({"threshold": float(threshold)}, open("out/03_02_threshold.json", "w"))
check_03_02()

`clean` comes from `spamtools.py`, a file, not from this notebook, because a saved pipeline records the
functions it uses by name; the checkpoint has to be able to find `spamtools.clean` to load it.

## 7. Exit ticket

**x1.** A cancer screen finds 50 of 250 patients who have cancer. What is its recall? (a) 0.5, (b) 0.2, (c) 0.25

**x2.** What does AUC measure? (a) the chance a random spam scores above a random ham, (b) accuracy at
threshold 0.5, (c) the best F1 over all thresholds

In [ ]:
ask("x1", "")
ask("x2", "")